In [26]:
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torchvision import models, transforms
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from tqdm import tqdm
import numpy as np
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from catboost import CatBoostClassifier

warnings.filterwarnings("ignore")
tqdm.pandas()

In [27]:
DEVICE = "mps" if torch.mps.is_available() else "cpu" 

DATA_FOLDER = Path(r"/Users/semyonkuricyn/vs/RWB_ML/ML_solve/Semøn/data/competition")
IMAGE_FOLDER = Path(r"/Users/semyonkuricyn/vs/RWB_ML/ML_solve/Semøn/data/images")

print(DEVICE)

mps


In [ ]:
res_net = models.resnet152(weights=models.ResNet152_Weights.IMAGENET1K_V1).to(DEVICE)

Downloading: "https://download.pytorch.org/models/resnet152-394f9c45.pth" to /Users/semyonkuricyn/.cache/torch/hub/checkpoints/resnet152-394f9c45.pth


100%|██████████| 230M/230M [00:34<00:00, 6.98MB/s] 


In [213]:
vit = models.vit_b_16(weights='IMAGENET1K_V1') 

Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to /Users/semyonkuricyn/.cache/torch/hub/checkpoints/vit_b_16-c867db91.pth


100%|██████████| 330M/330M [00:53<00:00, 6.46MB/s] 


In [214]:
vit = vit.to(DEVICE)

In [6]:
train_df = pd.read_csv(DATA_FOLDER / "train.csv")
test_df = pd.read_csv(DATA_FOLDER / "test.csv")

In [ ]:
tr_lst = train_df.to_numpy().tolist()
ts_lst = test_df.to_numpy().tolist()
tr_c = [el[1] for el in tr_lst]
ts_c = [el[1] for el in ts_lst]
print(len(set(tr_c) & set(ts_c)))

0


In [34]:
class ImageDataset(Dataset):
    def __init__(self, root_dir, df):
        self.samples = [str(root_dir) + "/" + str(i) + '.jpg' for i in df['id'].tolist()]

    def __len__(self):  
        return len(self.samples)

    def __getitem__(self, idx):
        img_path = self.samples[idx]
        img = Image.open(img_path).convert('RGB')
        w, h = img.size if isinstance(img, Image.Image) else (img.shape[2], img.shape[1])
        min_side = min(w, h)
        img_cropped = transforms.CenterCrop(min_side)(img)
        img_resized = transforms.Resize((256, 256))(img_cropped)
        img_tensor = transforms.ToTensor()(img_resized)
        
        return img_tensor

In [221]:
class ImageDataset_2(Dataset):
    def __init__(self, root_dir, df):
        self.samples = [str(root_dir) + "/" + str(i) + '.jpg' for i in df['id'].tolist()]

    def __len__(self):  
        return len(self.samples)

    def __getitem__(self, idx):
        img_path = self.samples[idx]
        img = Image.open(img_path).convert('RGB')
        w, h = img.size if isinstance(img, Image.Image) else (img.shape[2], img.shape[1])
        min_side = min(w, h)
        img_cropped = transforms.CenterCrop(min_side)(img)
        img_resized = transforms.Resize((224, 224))(img_cropped)
        img_tensor = transforms.ToTensor()(img_resized)
        img_recolored = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])(img_tensor)
        
        return img_recolored

In [35]:
tr_images_ds = ImageDataset(IMAGE_FOLDER, train_df)

batch_size = 32

images = DataLoader(tr_images_ds, batch_size=batch_size, shuffle=False)

In [222]:
tr_images_ds2 = ImageDataset_2(IMAGE_FOLDER, train_df)

images2 = DataLoader(tr_images_ds2, batch_size=batch_size, shuffle=False)

In [36]:
sureness = []

for batch in tqdm(images):
    with torch.no_grad():
        image_sure = res_net(batch.to(DEVICE))

    sure = image_sure.to('cpu').numpy()

    sureness.extend(sure)

100%|██████████| 1065/1065 [16:29<00:00,  1.08it/s]


In [223]:
sureness2 = []

for batch in tqdm(images2):
    with torch.no_grad():
        image_sure = vit(batch.to(DEVICE))

    sure = image_sure.to('cpu').numpy()

    sureness2.extend(sure)

100%|██████████| 1065/1065 [18:07<00:00,  1.02s/it]


In [224]:
DATA_PATH = "/Users/semyonkuricyn/vs/RWB_ML/ML_solve/Semøn/data"

train_text_emb = np.load(f'{DATA_PATH}/train_text_embeddings.npy', allow_pickle=True).tolist()
train_img_emb = np.load(f'{DATA_PATH}/train_img_embedings.npy', allow_pickle=True).tolist()

test_text_emb = np.load(f'{DATA_PATH}/test_text_embeddings.npy', allow_pickle=True).tolist()
test_img_emb = np.load(f'{DATA_PATH}/test_img_embedings.npy', allow_pickle=True).tolist()

In [225]:
for i in range(len(train_text_emb[-1])):
    train_df[f"text{i}"] = [train_text_emb[j][i] for j in range(len(train_text_emb))]

for i in range(len(train_img_emb[-1])):
    train_df[f"img{i}"] = [train_img_emb[j][i] for j in range(len(train_img_emb))]

for i in range(1000):
    train_df[f"f{i}"] = [sureness2[j][i] for j in range(len(sureness))]

In [ ]:
# model = CatBoostClassifier(
#     iterations=2000,           # Trees; sufficient for ~3000 samples (stop early if needed)
#     learning_rate=0.05,        # Conservative for stability with embeddings
#     depth=8,                   # Handles 3000 dims without excessive complexity
#     l2_leaf_reg=5,             # Regularization to prevent overfitting high-dim data
#     bootstrap_type='Bernoulli',
#     subsample=0.8,             # Adds randomness, good for embeddings
#     random_strength=1.0,       # Strengthens randomization
#     eval_metric='AUC',         # Or 'Logloss' for classification; monitor on validation
#     early_stopping_rounds=200, # Prevents overtraining
#     verbose=100                # Log every 100 trees
# )

# model = CatBoostClassifier(
#     iterations=1000,
#     learning_rate=0.1,
#     depth=6,
#     l2_leaf_reg=10,
#     bootstrap_type='Bernoulli',
#     subsample=0.7,
#     rsm=0.1,
#     random_strength=3.0,
#     early_stopping_rounds=100,
#     eval_metric='AUC'  
# )

# model = CatBoostClassifier(
#     iterations=3000,
#     learning_rate=0.03,
#     depth=7,
#     l2_leaf_reg=15,      # Higher for embeddings [web:17]
#     border_count=64,     # Reduces splits for numerical features
#     bootstrap_type='MVS', # More robust for high-dim noise
#     bagging_temperature=2.0,
#     early_stopping_rounds=300,
#     eval_metric="AUC"
# )

model = CatBoostClassifier( #very good AUC ~0.95
    iterations=5000,
    learning_rate=0.02,
    depth=6,
    l2_leaf_reg=20,
    border_count=128,
    bootstrap_type='Bernoulli',
    subsample=0.66,
    random_strength=2,
    loss_function='Logloss',
    eval_metric='AUC',
    custom_metric='AUC:hints=skip_train~false',
    od_type='Iter',
    od_wait=500,
    use_best_model=True
)

# catboost_model = CatBoostClassifier(
#     iterations=5000,
#     learning_rate=0.01,
#     depth=6,                  # Deeper, but heavily regularized
#     l2_leaf_reg=20,
#     rsm=0.2,                  # Only look at 600 features per split!
#     random_strength=5.0,      # Heavy penalty for "too perfect" splits
#     bagging_temperature=2.0,  # Aggressive row subsampling
#     min_data_in_leaf=50,      # Prevents tiny leaves fitting to noise
#     early_stopping_rounds=100,
#     eval_metric='AUC',
#     verbose=200,
#     text_features=['description'],
#     random_state=67,
#     leaf_estimation_method='Newton'   
# )

# model = CatBoostClassifier(
#     iterations=3000,
#     learning_rate=0.05,
#     depth=8,
#     l2_leaf_reg=8,
#     bootstrap_type='MVS',
#     rsm=0.8,                 # 80% feature subspace per tree [web:13]
#     random_strength=1.5,
#     border_count=64,
#     grow_policy='SymmetricTree',
#     eval_metric='AUC',
#     early_stopping_rounds=200,
#     thread_count=-1,         # All CPU cores [web:23]
#     task_type='CPU'
# )


model = CatBoostClassifier(
    iterations=4000,          # Компромисс между 5000 и 3000
    learning_rate=0.03,       # Среднее: стабильность + скорость
    depth=7,                  # Баланс между 6 и 8
    l2_leaf_reg=12,           # Усреднённая сильная регуляризация
    border_count=96,          # Компромисс 128/64 для high-dim
    bootstrap_type='MVS',     # Из high-precision (лучше для noisy embeddings)
    rsm=0.75,                 # Subspace из high-precision (75% features/tree)
    subsample=0.7,            # Из conservative
    random_strength=1.8,      # Компромисс randomization
    bagging_temperature=1.0,  # Баланс шума и стабильности
    loss_function='Logloss',
    eval_metric='AUC',
    custom_metric='AUC:hints=skip_train~false',
    early_stopping_rounds=300, # Компромисс 500/200
    grow_policy='SymmetricTree',
    use_best_model=True,
    verbose=200
)

In [227]:
features = [f'f{i}' for i in range(1000)] + [f'text{j}' for j in range(len(train_text_emb[0]))] #+ [f'img{k}' for k in range(len(train_img_emb[-1]))]
target = 'label'

In [228]:
X = train_df[features]
y = train_df[target]

In [229]:
X1, X2, y1, y2 = train_test_split(X, y, shuffle=True, random_state=52, stratify=y, test_size=0.179)


In [230]:
model.fit(X1, y1, eval_set=(X2, y2))

pred_acc = model.predict(X2)
print("Точность на валидационной выборке:", accuracy_score(pred_acc, y2))

pred_auc = model.predict_proba(X2)[:, 1]
print("ROC AUC на валидационной выборке:", roc_auc_score(y2, pred_auc))

0:	learn: 0.8573347	test: 0.8503480	best: 0.8503480 (0)	total: 82.9ms	remaining: 5m 31s
200:	learn: 0.9442255	test: 0.9226000	best: 0.9226000 (200)	total: 10.1s	remaining: 3m 10s
400:	learn: 0.9655551	test: 0.9334662	best: 0.9334662 (400)	total: 20.2s	remaining: 3m 1s
600:	learn: 0.9804034	test: 0.9412119	best: 0.9412330 (599)	total: 30.1s	remaining: 2m 50s
800:	learn: 0.9889843	test: 0.9450895	best: 0.9450952 (798)	total: 40.1s	remaining: 2m 40s
1000:	learn: 0.9940013	test: 0.9474692	best: 0.9474692 (1000)	total: 50.1s	remaining: 2m 30s
1200:	learn: 0.9967654	test: 0.9488883	best: 0.9488883 (1200)	total: 1m	remaining: 2m 20s
1400:	learn: 0.9983536	test: 0.9502748	best: 0.9503082 (1390)	total: 1m 10s	remaining: 2m 10s
1600:	learn: 0.9991842	test: 0.9511486	best: 0.9511493 (1599)	total: 1m 21s	remaining: 2m 1s
1800:	learn: 0.9996349	test: 0.9518561	best: 0.9518687 (1799)	total: 1m 31s	remaining: 1m 51s
2000:	learn: 0.9998299	test: 0.9526196	best: 0.9526490 (1993)	total: 1m 42s	remaining

In [198]:
ts_images_ds = ImageDataset(IMAGE_FOLDER, test_df)

In [199]:
images_ts = DataLoader(ts_images_ds, batch_size=batch_size, shuffle=False)

In [201]:
sureness_ts = []

for images_ in tqdm(images_ts):
    with torch.no_grad():
        sure = res_net(images_.to(DEVICE))

    sureness_ts.extend(sure.to('cpu').numpy().tolist())

100%|██████████| 264/264 [04:01<00:00,  1.09it/s]


In [231]:
for i in range(1000):
    test_df[f"f{i}"] = [sureness_ts[j][i] for j in range(len(sureness_ts))]

for i in range(len(train_text_emb[-1])):
    test_df[f"text{i}"] = [test_text_emb[j][i] for j in range(len(test_text_emb))]

for i in range(len(train_img_emb[-1])):
    test_df[f"img{i}"] = [test_img_emb[j][i] for j in range(len(test_img_emb))]

In [232]:
pred_ts = model.predict_proba(test_df[features])

In [233]:
test_df['y_pred'] = [i[1] for i in pred_ts.tolist()]

In [234]:
(test_df[["id", 'y_pred']]).to_csv("submission_resnet+emb+catboost.csv", index=False)